# EAPS 507: Parallel Computing in Python
## Threads, processes, Joblib, MPI, and `simplempi`

This notebook introduces several ways to run independent scientific tasks at the same time. The examples use small synthetic geoscience workloads, but the same patterns apply to processing model files, ensemble members, years, storms, stations, and spatial tiles.

**Estimated time:** 2--2.5 hours  
**Main goal:** choose a parallel tool because it matches the problem, not simply because it is available.

## Learning objectives

By the end of this activity, you should be able to:

1. Distinguish concurrency from parallelism.
2. Explain the difference among threads, processes, and MPI ranks.
3. Match I/O-bound and CPU-bound tasks to suitable tools.
4. Parallelize an independent loop with `concurrent.futures` and Joblib.
5. Explain `rank`, `size`, `scatter`, `gather`, and `reduce` in MPI.
6. Use `simplempi.parfor` for an embarrassingly parallel loop.
7. Recognize overhead, oversubscription, shared-state problems, and reproducibility issues.

## 1. The central idea

A serial program completes one task after another. A parallel program divides work among multiple workers. Parallel computing is most useful when the tasks are sufficiently expensive and can be completed with limited communication.

| Approach | Workers share memory? | Best starting use | Typical Python tool |
|---|---:|---|---|
| Threads | Yes | Waiting on files, servers, or other I/O | `ThreadPoolExecutor` |
| Processes | No, by default | CPU-heavy independent calculations | `ProcessPoolExecutor`, Joblib |
| MPI ranks | No; communicate explicitly | Multi-node clusters and distributed-memory work | `mpi4py`, `simplempi` |

**Concurrency** means tasks make progress during overlapping periods. 
For instance:
- For I/O-bound tasks (such as waiting for a web request, file download, or database query), threads release the GIL while waiting, letting other threads make progress.
- For CPU-bound tasks (heavy math or data processing), the GIL forces threads to take turns on a single core, meaning multithreading does not speed up execution.
**Parallelism** means tasks actually execute at the same time. Threads can provide concurrency even when only one thread is executing Python bytecode at a given instant.

In [1]:
## LOAD PACKAGES
import os
import platform
import time
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr

print(f"Python: {platform.python_version()}")  #prints the python environment you are using
print(f"Logical CPUs visible to this session: {os.cpu_count()}")  #prints the number of CPUs visible to the session
print(f"Current process ID: {os.getpid()}")  #The current process ID

Python: 3.13.15
Logical CPUs visible to this session: 128
Current process ID: 2118483


### Resource rule

On a shared system, `os.cpu_count()` may report more CPUs than your job allocation. Use only the resources requested from the scheduler. For example, if Slurm gives your job four CPUs, use no more than four process workers. More workers can increase memory use and context switching without making the code faster.

## 2. Begin with a serial baseline

Never evaluate parallel speed without first timing correct serial code. The following function represents a CPU-bound diagnostic applied independently to several ensemble members. It intentionally uses a Python loop so that the cost is easy to observe.

In [2]:
def member_diagnostic(member, n=700_000):
    total = 0.0
    for i in range(1, n):
        total += ((i + member) % 997) ** 0.5
    return {"member": member, "diagnostic": total / n, "pid": os.getpid()}

members = list(range(8))
start = time.perf_counter()
serial_results = [member_diagnostic(member) for member in members]
serial_time = time.perf_counter() - start

print(f"Serial time: {serial_time:.2f} s")
pd.DataFrame(serial_results).head()

Serial time: 0.64 s


,member,diagnostic,pid
0,0,21.032009,2118483
1,1,21.032022,2118483
2,2,21.032035,2118483
3,3,21.032047,2118483
4,4,21.032059,2118483


**Before continuing:** Which parts of this loop are independent? What information must each worker receive? What must be returned?

## 3. Threads: useful when tasks spend time waiting

Threads run inside one process and share its memory. They are often effective for I/O-bound work, such as reading several independent files. In standard CPython, the Global Interpreter Lock limits simultaneous execution of Python bytecode, so threads often do **not** accelerate a pure-Python CPU loop. Some compiled libraries, including parts of NumPy, can release the lock. Therefore, measure rather than assume.

The next example uses `sleep` to imitate file or network waiting without requiring external data.

In [3]:
from concurrent.futures import ThreadPoolExecutor

def pretend_file_read(filename, delay=0.35):
    '''This is a test pretend reading file program. In this function, we deliberately cause the process to sleep for 0.35s.
    What this would show us is that for a serial process, it will take 0.35 * the number of processess, whereas for a 
    thread process, it would take us 0.35 * number of processess / number of threads or workers.'''
    time.sleep(delay)
    return {"file": filename, "thread_process": os.getpid()}  

files = [f"temperature_{year}.nc" for year in range(2000, 2008)]

start = time.perf_counter()
serial_io = [pretend_file_read(filename) for filename in files]
serial_io_time = time.perf_counter() - start

start = time.perf_counter()
with ThreadPoolExecutor(max_workers=4) as executor:   #feel free to change the number of max_workers and see how much more time it takes
    threaded_io = list(executor.map(pretend_file_read, files))
thread_io_time = time.perf_counter() - start

print(f"Serial waiting example:   {serial_io_time:.2f} s")
print(f"Four-thread example:      {thread_io_time:.2f} s")
print(f"Observed speedup:         {serial_io_time / thread_io_time:.2f}x")

Serial waiting example:   2.80 s
Four-thread example:      0.70 s
Observed speedup:         3.99x


### Test the wrong tool on purpose

Run the CPU-bound diagnostic with threads. Record the time and compare it with the serial baseline. A lack of speedup is a useful result, not a failed experiment.

In [4]:
start = time.perf_counter()
with ThreadPoolExecutor(max_workers=4) as executor:
    threaded_cpu_results = list(executor.map(member_diagnostic, members))
thread_cpu_time = time.perf_counter() - start

print(f"Serial CPU time:   {serial_time:.2f} s")
print(f"Threaded CPU time: {thread_cpu_time:.2f} s")
print(f"Speedup:           {serial_time / thread_cpu_time:.2f}x")

Serial CPU time:   0.64 s
Threaded CPU time: 0.74 s
Speedup:           0.87x


## 4. Processes: separate Python interpreters

Processes have separate memory spaces and separate Python interpreters. This makes them suitable for independent CPU-bound Python work, but arguments and results must usually be serialized. Starting workers and moving data have costs.

Notebook-defined functions can be difficult to import in new worker processes on some systems. We place the worker function in a small module, which is also a good scientific-programming practice. In a standalone Python script, create process pools inside an `if __name__ == "__main__":` block so that child processes do not recreate the pool.

In [5]:
helper_code = '''import os

def member_diagnostic(member, n=700_000):
    total = 0.0
    for i in range(1, n):
        total += ((i + member) % 997) ** 0.5
    return {"member": member, "diagnostic": total / n, "pid": os.getpid()}
'''
Path("parallel_helpers.py").write_text(helper_code)
print("Created parallel_helpers.py")

Created parallel_helpers.py


In [6]:
from concurrent.futures import ProcessPoolExecutor
from parallel_helpers import member_diagnostic as process_diagnostic

start = time.perf_counter()
with ProcessPoolExecutor(max_workers=4) as executor:
    process_results = list(executor.map(process_diagnostic, members))
process_time = time.perf_counter() - start

print(f"Serial time:       {serial_time:.2f} s")
print(f"Four-process time: {process_time:.2f} s")
print(f"Speedup:           {serial_time / process_time:.2f}x")
pd.DataFrame(process_results)

Serial time:       0.64 s
Four-process time: 0.20 s
Speedup:           3.26x


,member,diagnostic,pid
0,0,21.032009,2118550
1,1,21.032022,2118551
2,2,21.032035,2118552
3,3,21.032047,2118553
4,4,21.032059,2118553
5,5,21.032071,2118552
6,6,21.032083,2118551
7,7,21.032094,2118550


### Optional environment check

The local sections require NumPy, pandas, xarray, and Joblib. The MPI sections additionally require a system MPI implementation plus `mpi4py`; `simplempi` is needed for the `parfor` example. On a managed cluster, use the course environment or site-provided modules instead of installing a second MPI implementation yourself.

```bash
python -m pip install numpy pandas xarray dask joblib
# Only when consistent with your system's MPI instructions:
python -m pip install mpi4py simplempi
```

### Checkpoint questions

1. Why do the process IDs differ in the process example but not the thread example?
2. Was the process speedup exactly 4x? Explain at least two reasons why it may not be.
3. What would happen if each task lasted only one millisecond?
4. Why is sending a multi-gigabyte xarray object to every process potentially expensive?

## 5. Joblib: a concise parallel loop

Joblib provides a readable pattern for applying a function to many arguments. Its default process-based backend is called `loky`. Set `n_jobs=1` while debugging, then increase the worker count after confirming correctness. `n_jobs=-1` requests all detected CPUs, which may be inappropriate on a shared cluster.

In [7]:
try:
    from joblib import Parallel, delayed
    JOBLIB_AVAILABLE = True
except ImportError:
    JOBLIB_AVAILABLE = False
    print("Joblib is not installed. Use: python -m pip install joblib")

In [8]:
if JOBLIB_AVAILABLE:
    start = time.perf_counter()
    joblib_results = Parallel(n_jobs=4, backend="loky", verbose=5)(
        delayed(process_diagnostic)(member) for member in members
    )
    joblib_time = time.perf_counter() - start
    print(f"Joblib time: {joblib_time:.2f} s")
    print(f"Speedup:     {serial_time / joblib_time:.2f}x")
    display(pd.DataFrame(joblib_results))

[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.


Joblib time: 0.40 s
Speedup:     1.59x


[Parallel(n_jobs=4)]: Done   3 out of   8 | elapsed:    0.3s remaining:    0.5s
[Parallel(n_jobs=4)]: Done   5 out of   8 | elapsed:    0.4s remaining:    0.2s
[Parallel(n_jobs=4)]: Done   8 out of   8 | elapsed:    0.4s finished


,member,diagnostic,pid
0,0,21.032009,2118565
1,1,21.032022,2118563
2,2,21.032035,2118564
3,3,21.032047,2118566
4,4,21.032059,2118564
5,5,21.032071,2118563
6,6,21.032083,2118565
7,7,21.032094,2118566


### Joblib backend experiment

Change `backend="loky"` to `backend="threading"`. Predict the result first, then run it. The best backend depends on whether the called function spends most of its time in Python, in compiled code that releases the GIL, or waiting for I/O.

## 6. A geoscience example: independent station files

The pattern below creates one synthetic temperature series per station and computes a 95th-percentile threshold and the number of exceedance days. Each station can be analyzed independently. In a real workflow, the worker might open one station file, one year, or one ensemble member.

In [9]:
station_code = '''import os
import numpy as np

def analyze_station(station_id, n_days=3650):
    rng = np.random.default_rng(10_000 + station_id)
    day = np.arange(n_days)
    temperature = 25 + 7 * np.sin(2 * np.pi * day / 365.25) + rng.normal(0, 3, n_days)
    threshold = np.quantile(temperature, 0.95)
    thresh_90 = np.quantile(temperature, 0.90)
    mean_temp = np.mean(temperature)
    days_over_90 = int((temperature > thresh_90).sum())
    return {"station": station_id, "p95": threshold, "mean_temp": mean_temp, "95_exceedance_days": int((temperature > threshold).sum()), "90_exceedance_days": days_over_90, "pid": os.getpid()}
'''
Path("station_analysis.py").write_text(station_code)
from station_analysis import analyze_station

station_ids = list(range(24))
station_results = Parallel(n_jobs=4)(delayed(analyze_station)(station) for station in station_ids) if JOBLIB_AVAILABLE else [analyze_station(station) for station in station_ids]
station_df = pd.DataFrame(station_results)
station_df.head()

,station,p95,mean_temp,95_exceedance_days,90_exceedance_days,pid
0,0,34.284390,25.002089,183,365,2118564
1,1,34.205339,24.992983,183,365,2118563
2,2,34.108871,24.985914,183,365,2118565
3,3,34.082976,25.024756,183,365,2118566
4,4,34.087232,24.938711,183,365,2118564


### Student task

Modify `analyze_station` to also return the mean temperature and the longest run of days above the 90th percentile. Confirm that results are identical for `n_jobs=1` and `n_jobs=4`. Then explain why identical answers matter more than speedup.

## 7. Where xarray and Dask fit

Xarray describes labeled array operations. Dask can divide those arrays into chunks and schedule chunk-level work using threads, processes, or a distributed scheduler. Calling `.compute()` triggers the deferred calculation. Avoid wrapping a Dask-parallel xarray calculation inside another process pool unless you have intentionally configured nested parallelism.

In [10]:
rng = np.random.default_rng(507)
temperature = xr.DataArray(
    rng.normal(size=(365, 60, 120)),
    dims=("time", "lat", "lon"),
    coords={"time": pd.date_range("2001-01-01", periods=365), "lat": np.linspace(-30, 30, 60), "lon": np.linspace(0, 359, 120)},
    name="temperature_anomaly",
)

try:
    chunked = temperature.chunk({"time": 30, "lat": 20, "lon": 40})
    print(chunked)
    seasonal_std = chunked.groupby("time.season").std("time")
    print("Before compute:", type(seasonal_std.data))
    seasonal_std = seasonal_std.compute()
    print("After compute: ", type(seasonal_std.data))
except ImportError:
    print("Dask is not installed; the unchunked xarray object still works serially.")

<xarray.DataArray 'temperature_anomaly' (time: 365, lat: 60, lon: 120)> Size: 21MB
dask.array<xarray-<this-array>, shape=(365, 60, 120), dtype=float64, chunksize=(30, 20, 40), chunktype=numpy.ndarray>
Coordinates:
  * time     (time) datetime64[us] 3kB 2001-01-01 2001-01-02 ... 2001-12-31
  * lat      (lat) float64 480B -30.0 -28.98 -27.97 -26.95 ... 27.97 28.98 30.0
  * lon      (lon) float64 960B 0.0 3.017 6.034 9.05 ... 349.9 353.0 356.0 359.0
Before compute: <class 'dask.array.core.Array'>
After compute:  <class 'numpy.ndarray'>


## 8. MPI: distributed-memory parallelism

MPI launches multiple independent processes called **ranks**. Every rank runs the same program. `rank` identifies the current worker and `size` gives the total number of workers in a communicator.

Common communication patterns:

- `broadcast`: one rank sends the same information to every rank.
- `scatter`: divide a collection among ranks.
- `gather`: collect rank-specific results on one rank.
- `reduce`: combine values using an operation such as sum, minimum, or maximum.
- `send` and `recv`: direct point-to-point communication.

MPI programs should normally be saved as scripts and launched outside the notebook kernel.

In [11]:
mpi_code = '''from mpi4py import MPI
import numpy as np

comm = MPI.COMM_WORLD
rank = comm.Get_rank()
size = comm.Get_size()

# Each rank creates a reproducible local precipitation sample.
rng = np.random.default_rng(507 + rank)
local_precip = rng.gamma(shape=2.0, scale=4.0, size=10_000)
local_sum = local_precip.sum()
local_count = local_precip.size

global_sum = comm.reduce(local_sum, op=MPI.SUM, root=0)
global_count = comm.reduce(local_count, op=MPI.SUM, root=0)

print(f"rank {rank}/{size}: local mean = {local_precip.mean():.3f}")
if rank == 0:
    print(f"Global mean = {global_sum / global_count:.3f}")
'''
Path("mpi_precip_mean.py").write_text(mpi_code)
print("Created mpi_precip_mean.py")
print("Run in a terminal with: mpiexec -n 4 python mpi_precip_mean.py")

Created mpi_precip_mean.py
Run in a terminal with: mpiexec -n 4 python mpi_precip_mean.py


### Run the MPI example

Use one of the following commands in a terminal after activating the correct environment:

```bash
mpiexec -n 4 python mpi_precip_mean.py
# or, depending on the MPI installation
mpirun -n 4 python mpi_precip_mean.py
# on a Slurm cluster, after requesting resources
srun -n 4 python mpi_precip_mean.py
```

The printed rank order is not guaranteed. The collective `reduce` still gives a well-defined global result.

## 9. `simplempi`: a smaller interface for independent loops

The `simplempi.parfor` interface distributes an iterable across MPI ranks. It is useful when each iteration is independent and little inter-rank communication is needed. Because all ranks execute the script, avoid having every rank write to the same output file. Give each task or rank a unique filename, or collect results deliberately.

In [12]:
simplempi_code = '''from pathlib import Path
import numpy as np
from simplempi.parfor import parfor, pprint

output_dir = Path("simplempi_output")
output_dir.mkdir(exist_ok=True)
years = list(range(2000, 2012))

for year in parfor(years):
    rng = np.random.default_rng(year)
    precip = rng.gamma(shape=2.0, scale=4.0, size=365)
    summary = np.array([year, precip.mean(), np.quantile(precip, 0.95)])
    np.savetxt(output_dir / f"precip_summary_{year}.csv", summary[None, :], delimiter=",", header="year,mean,p95", comments="")
    pprint(f"completed {year}")
'''
Path("simplempi_years.py").write_text(simplempi_code)
print("Created simplempi_years.py")
print("Run in a terminal with: mpiexec -n 4 python simplempi_years.py")

Created simplempi_years.py
Run in a terminal with: mpiexec -n 4 python simplempi_years.py


### MPI and `simplempi` comparison

| Question | `mpi4py` | `simplempi.parfor` |
|---|---|---|
| How much control do I have? | High | Deliberately limited |
| Can I use scatter/gather/reduce directly? | Yes | Use `mpi4py` when those patterns are needed |
| Is it concise for independent iterations? | Moderate | Yes |
| Good first use | Distributed algorithms and explicit communication | Independent years, files, cases, or ensemble members |

## 10. A generic Slurm job script

Cluster module names, accounts, partitions, and launch commands differ. Treat this as a template and adapt it to the instructions for your system.

In [13]:
slurm_script = '''#!/bin/bash
#SBATCH --job-name=eaps507_mpi
#SBATCH --nodes=1
#SBATCH --ntasks=4
#SBATCH --cpus-per-task=1
#SBATCH --time=00:10:00
#SBATCH --mem=4G
#SBATCH --output=eaps507_mpi_%j.out

# Activate the course environment here.
# module load <mpi-module>
# source <environment-activation-command>

srun -n 4 python mpi_precip_mean.py
'''
Path("parallel_job.slurm").write_text(slurm_script)
print("Created parallel_job.slurm")
print("Submit with: sbatch parallel_job.slurm")

Created parallel_job.slurm
Submit with: sbatch parallel_job.slurm


## 11. Common mistakes

1. **Parallelizing before verifying correctness.** Compare parallel output with a serial reference.
2. **Using too many workers.** Match workers to the allocation and available memory.
3. **Parallelizing tiny tasks.** Worker startup, scheduling, and data transfer can dominate.
4. **Sending large arrays repeatedly.** Open or generate data inside the worker when appropriate, or use chunked/shared-memory approaches.
5. **Writing one file from many workers.** Use unique files or a controlled gather/write step.
6. **Assuming output order.** Parallel completion order may differ from input order. Preserve identifiers.
7. **Nested parallelism.** Four processes that each start eight BLAS threads may create 32 active threads.
8. **Non-reproducible random numbers.** Give each task a deterministic, independent seed.
9. **Ignoring memory.** Separate processes can duplicate large objects.
10. **Testing only once.** Timings vary; repeat measurements and report the spread.

## 12. Choosing a tool

| Situation | Reasonable first choice |
|---|---|
| Reading several independent files | Threads, xarray/Dask, or careful multi-file I/O |
| Pure-Python CPU calculation across independent cases | Processes or Joblib `loky` |
| NumPy-heavy calculation | Test threads and processes; compiled code may release the GIL |
| Labeled arrays already chunked with Dask | Let Dask schedule the chunks |
| Work must span cluster nodes | MPI or a distributed scheduler |
| Independent loop with minimal communication on an MPI system | `simplempi.parfor` |
| Complex rank-to-rank communication | `mpi4py` |

The simplest correct tool is usually the best starting point.

## 13. Final in-class challenge

Choose **one** unit of parallel work: station, year, ensemble member, storm, model, or file. Then:

1. Write a serial function that accepts one task identifier and returns a small result.
2. Test it on one task.
3. Run all tasks serially and record the time.
4. Parallelize with Joblib or `ProcessPoolExecutor`.
5. Verify that serial and parallel results agree.
6. Repeat with 1, 2, and 4 workers.
7. Create a small table of runtime and speedup.
8. Explain whether the observed scaling is reasonable.

**Deliverable:** code, timing table, one figure, and a 150-word interpretation.

In [14]:
# Starter structure for the final challenge
def analyze_one_task(task_id):
    # TODO: calculate and return a small result for one independent task.
    raise NotImplementedError

tasks = []  # TODO: add task identifiers
worker_counts = [1, 2, 4]
timings = []

# TODO: run with each worker count, verify the results, and fill timings.
# timing_df = pd.DataFrame(timings)
# timing_df["speedup"] = timing_df.loc[timing_df.workers == 1, "seconds"].iloc[0] / timing_df["seconds"]
# timing_df

## 14. Exit ticket

Answer in 3--5 sentences each:

1. Why can threads accelerate file waiting but fail to accelerate a pure-Python loop?
2. What is one cost introduced by processes?
3. What is the difference between an MPI rank and a thread?
4. Give one EAPS research task that is embarrassingly parallel.
5. What evidence would convince you that parallelization was successful?

## References and further reading

- Python `concurrent.futures`: https://docs.python.org/3/library/concurrent.futures.html
- Python `multiprocessing`: https://docs.python.org/3/library/multiprocessing.html
- Joblib `Parallel`: https://joblib.readthedocs.io/en/stable/generated/joblib.Parallel.html
- mpi4py documentation: https://mpi4py.readthedocs.io/en/stable/
- mpi4py tutorial: https://mpi4py.readthedocs.io/en/stable/tutorial.html
- `simplempi` package and examples: https://pypi.org/project/simplempi/
- Xarray with Dask: https://docs.xarray.dev/en/stable/user-guide/dask.html